In [16]:
import csv

guides = []
with open("results/guides_spcas9.csv") as f:
    for row in csv.DictReader(f):
        guides.append(row)

print(len(guides))

27


In [17]:
print(guides[0])

{'variant': 'NM_000202.8(IDS):c.1400C>T (p.Pro467Leu)', 'editor': 'ABE', 'edited_strand': '+', 'grch38_position': '149482999', 'reference': 'G', 'variant_allele': 'A', 'protospacer': 'AGGCCGGGGATACTGGCTAT', 'pam': 'AGG', 'window_position': '8', 'bystanders': '0', 'clean': 'yes'}


In [18]:
for guide in guides:
    guide["window_position"] = int(guide["window_position"])
    guide["bystanders"] = int(guide["bystanders"])

print(guides[0]["window_position"], guides[0]["bystanders"])

8 0


In [19]:
def bystander_positions(guide):
    if guide["editor"] == "ABE":
        target_base = "A"
    else:
        target_base = "C"

    positions = []
    for p in [4, 5, 6, 7, 8]:
        if p == guide["window_position"]:
            continue
        if guide["protospacer"][p - 1] == target_base:
            positions.append(p)
    return positions

print(bystander_positions(guides[0]))

[]


In [20]:
same = 0
for guide in guides:
    found = len(bystander_positions(guide))
    if found == guide["bystanders"]:
        same = same + 1
    else:
        print("MISMATCH:", guide["variant"], found, guide["bystanders"])

print(same, "of", len(guides), "agree")

27 of 27 agree


In [21]:
def cdna_position(name):
    start = name.find("c.") + 2
    number = ""
    for character in name[start:]:
        if character.isdigit():
            number = number + character
        elif character == "+" or character == "-":
            return None
        else:
            break
    return int(number)

print(cdna_position("NM_000202.8(IDS):c.1400C>T (p.Pro467Leu)"))
print(cdna_position("NM_000202.8(IDS):c.104-2A>G"))

1400
None


In [22]:
for guide in guides:
    guide["c_pos"] = cdna_position(guide["variant"])

print(guides[0]["c_pos"])

1400


In [23]:
def cdna_position_of(guide, p):
    offset = p - guide["window_position"]
    if guide["edited_strand"] == "-":
        return guide["c_pos"] + offset
    else:
        return guide["c_pos"] - offset

print(cdna_position_of(guides[0], 8))
print(cdna_position_of(guides[0], 7))
print(cdna_position_of(guides[0], 9))

1400
1401
1399


In [24]:
complement = {"A": "T", "T": "A", "C": "G", "G": "C"}

def coding_base(guide, p):
    base = guide["protospacer"][p - 1]
    if guide["edited_strand"] == "-":
        return base
    else:
        return complement[base]

print(coding_base(guides[0], 8))

C


In [25]:
def protospacer_position(guide, c):
    offset = c - guide["c_pos"]
    if guide["edited_strand"] == "-":
        return guide["window_position"] + offset
    else:
        return guide["window_position"] - offset

print(protospacer_position(guides[0], 1400))
print(protospacer_position(guides[0], 1399))

8
9


In [26]:
def read_codon(guide, codon_number):
    first = 3 * codon_number - 2
    codon = ""
    for c in [first, first + 1, first + 2]:
        p = protospacer_position(guide, c)
        if p < 1 or p > 20:
            return None
        codon = codon + coding_base(guide, p)
    return codon

print(read_codon(guides[0], 467))

CCC


In [27]:
codon_table = {
    "TTT": "Phe", "TTC": "Phe", "TTA": "Leu", "TTG": "Leu",
    "TCT": "Ser", "TCC": "Ser", "TCA": "Ser", "TCG": "Ser",
    "TAT": "Tyr", "TAC": "Tyr", "TAA": "Ter", "TAG": "Ter",
    "TGT": "Cys", "TGC": "Cys", "TGA": "Ter", "TGG": "Trp",
    "CTT": "Leu", "CTC": "Leu", "CTA": "Leu", "CTG": "Leu",
    "CCT": "Pro", "CCC": "Pro", "CCA": "Pro", "CCG": "Pro",
    "CAT": "His", "CAC": "His", "CAA": "Gln", "CAG": "Gln",
    "CGT": "Arg", "CGC": "Arg", "CGA": "Arg", "CGG": "Arg",
    "ATT": "Ile", "ATC": "Ile", "ATA": "Ile", "ATG": "Met",
    "ACT": "Thr", "ACC": "Thr", "ACA": "Thr", "ACG": "Thr",
    "AAT": "Asn", "AAC": "Asn", "AAA": "Lys", "AAG": "Lys",
    "AGT": "Ser", "AGC": "Ser", "AGA": "Arg", "AGG": "Arg",
    "GTT": "Val", "GTC": "Val", "GTA": "Val", "GTG": "Val",
    "GCT": "Ala", "GCC": "Ala", "GCA": "Ala", "GCG": "Ala",
    "GAT": "Asp", "GAC": "Asp", "GAA": "Glu", "GAG": "Glu",
    "GGT": "Gly", "GGC": "Gly", "GGA": "Gly", "GGG": "Gly",
}

print(codon_table["CCC"])

Pro


In [28]:
def coding_strand_edit(guide):
    if guide["editor"] == "ABE":
        if guide["edited_strand"] == "-":
            return "A", "G"
        else:
            return "T", "C"
    else:
        if guide["edited_strand"] == "-":
            return "C", "T"
        else:
            return "G", "A"

print(coding_strand_edit(guides[0]))

('T', 'C')


In [29]:
results = []

for guide in guides:
    for p in bystander_positions(guide):

        if guide["c_pos"] is None:
            results.append([guide["variant"], guide["protospacer"], None, "unassessed"])
            print(guide["variant"], "  intronic target, cannot assess")
            continue

        c = cdna_position_of(guide, p)
        codon_number = (c + 2) // 3
        before = read_codon(guide, codon_number)
        base_from, base_to = coding_strand_edit(guide)
        index = (c - 1) % 3

        if before[index] != base_from:
            print("PROBLEM at c." + str(c), "expected", base_from, "found", before[index])
            continue

        after = before[:index] + base_to + before[index + 1:]
        amino_before = codon_table[before]
        amino_after = codon_table[after]

        if amino_before == amino_after:
            consequence = "synonymous"
        elif amino_after == "Ter":
            consequence = "nonsense"
        else:
            consequence = "missense"

        results.append([guide["variant"], guide["protospacer"], c, consequence])
        print(guide["variant"])
        print("    c." + str(c), " ", before, "->", after, " ",
              amino_before, "->", amino_after, " ", consequence)

print()
print(len(results), "bystanders in total")

NM_000202.8(IDS):c.104-2A>G   intronic target, cannot assess
NM_000202.8(IDS):c.104-2A>G   intronic target, cannot assess
NM_000202.8(IDS):c.104-2A>G   intronic target, cannot assess
NM_000202.8(IDS):c.1181-1G>A   intronic target, cannot assess
NM_000202.8(IDS):c.1186C>T (p.Gln396Ter)
    c.1189   TCC -> CCC   Ser -> Pro   missense
NM_000202.8(IDS):c.1229T>C (p.Leu410Pro)
    c.1226   ACG -> ATG   Thr -> Met   missense
NM_000202.8(IDS):c.1229T>C (p.Leu410Pro)
    c.1228   CTG -> TTG   Leu -> Leu   synonymous
NM_000202.8(IDS):c.1327C>T (p.Arg443Ter)
    c.1330   TTC -> CTC   Phe -> Leu   missense
NM_000202.8(IDS):c.1327C>T (p.Arg443Ter)
    c.1326   TTT -> TTC   Phe -> Phe   synonymous
NM_000202.8(IDS):c.136G>A (p.Asp46Asn)
    c.137   GAC -> GGC   Asp -> Gly   missense
NM_000202.8(IDS):c.136G>A (p.Asp46Asn)
    c.137   GAC -> GGC   Asp -> Gly   missense
NM_000202.8(IDS):c.140T>C (p.Leu47Pro)
    c.138   GAC -> GAT   Asp -> Asp   synonymous
NM_000202.8(IDS):c.140T>C (p.Leu47Pro)
    c.1

In [30]:
clean_variants = []

for guide in guides:
    if guide["c_pos"] is None:
        continue

    all_synonymous = True
    for p in bystander_positions(guide):
        c = cdna_position_of(guide, p)
        codon_number = (c + 2) // 3
        before = read_codon(guide, codon_number)
        base_from, base_to = coding_strand_edit(guide)
        index = (c - 1) % 3
        after = before[:index] + base_to + before[index + 1:]
        if codon_table[before] != codon_table[after]:
            all_synonymous = False

    if all_synonymous:
        if guide["variant"] not in clean_variants:
            clean_variants.append(guide["variant"])

print(len(clean_variants), "variants have a guide that leaves the protein unchanged")
print()
for v in clean_variants:
    print(" ", v)

10 variants have a guide that leaves the protein unchanged

  NM_000202.8(IDS):c.1400C>T (p.Pro467Leu)
  NM_000202.8(IDS):c.1402C>T (p.Arg468Trp)
  NM_000202.8(IDS):c.223C>T (p.Gln75Ter)
  NM_000202.8(IDS):c.238C>T (p.Gln80Ter)
  NM_000202.8(IDS):c.263G>A (p.Arg88His)
  NM_000202.8(IDS):c.140T>C (p.Leu47Pro)
  NM_000202.8(IDS):c.1420C>T (p.Gln474Ter)
  NM_000202.8(IDS):c.1561G>A (p.Glu521Lys)
  NM_000202.8(IDS):c.1568A>G (p.Tyr523Cys)
  NM_000202.8(IDS):c.325T>C (p.Trp109Arg)
